# 0. Problem
## 1174. Immediate Food Delivery II — Medium
For each customer, take their first order. Calculate the percentage of those first orders that were immediate (`order_date = customer_pref_delivery_date`). Round to 2 decimals.

Official: https://leetcode.com/problems/immediate-food-delivery-ii/

# 1. Setup

In [ ]:
import pandas as pd
delivery_rows=[(1,1,"2019-08-01","2019-08-02"),(2,2,"2019-08-02","2019-08-02"),(3,1,"2019-08-11","2019-08-12"),(4,3,"2019-08-24","2019-08-24"),(5,3,"2019-08-21","2019-08-22"),(6,2,"2019-08-11","2019-08-13"),(7,4,"2019-08-09","2019-08-09")]
delivery_pd=pd.DataFrame(delivery_rows,columns=["delivery_id","customer_id","order_date","customer_pref_delivery_date"])
delivery_pd[["order_date","customer_pref_delivery_date"]]=delivery_pd[["order_date","customer_pref_delivery_date"]].apply(pd.to_datetime)
delivery_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
delivery_spark=(spark.createDataFrame(delivery_rows,["delivery_id","customer_id","order_date","customer_pref_delivery_date"]).withColumn("order_date",F.to_date("order_date")).withColumn("customer_pref_delivery_date",F.to_date("customer_pref_delivery_date")))
delivery_spark.createOrReplaceTempView("Delivery")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH first_order AS (
    SELECT customer_id, MIN(order_date) AS first_order_date
    FROM Delivery
    GROUP BY customer_id
)
SELECT ROUND(AVG(CASE WHEN d.order_date=d.customer_pref_delivery_date THEN 1.0 ELSE 0.0 END)*100,2) AS immediate_percentage
FROM Delivery d
JOIN first_order f
  ON d.customer_id=f.customer_id
 AND d.order_date=f.first_order_date
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
first_date=delivery_pd.groupby("customer_id")["order_date"].transform("min")
first_orders=delivery_pd.loc[delivery_pd["order_date"].eq(first_date)].copy()
result_pd=pd.DataFrame({"immediate_percentage":[round(first_orders["order_date"].eq(first_orders["customer_pref_delivery_date"]).mean()*100,2)]})
result_pd

# 4. PySpark Solution

In [ ]:
first_order=delivery_spark.groupBy("customer_id").agg(F.min("order_date").alias("first_order_date"))
first_orders=(delivery_spark.alias("d").join(first_order.alias("f"),(F.col("d.customer_id")==F.col("f.customer_id"))&(F.col("d.order_date")==F.col("f.first_order_date")),"inner"))
result_spark=first_orders.agg(F.round(F.avg(F.when(F.col("d.order_date")==F.col("d.customer_pref_delivery_date"),1.0).otherwise(0.0))*100,2).alias("immediate_percentage"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| first row/group | `MIN(date)` + join-back | `.transform("min")` + filter | group `min` + join-back |
| percentage | `AVG(CASE...)*100` | boolean `.mean()*100` | `avg(when(...))*100` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Delivery

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: delivery_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: delivery_spark